# Feature Selection

In [1]:
#general imports that we will need will almost always use - it is a good practice to import all libraries at the beginning of the notebook or script
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
sns.set()

# data partition
from sklearn.model_selection import train_test_split

#filter methods
# spearman 
# chi-square
import scipy.stats as stats
from scipy.stats import chi2_contingency

#wrapper methods
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.feature_selection import RFE


# embedded methods
from sklearn.linear_model import LassoCV

# ignore warnings
import warnings
warnings.filterwarnings('ignore')

#set random seed for reproducibility
RSEED = 42
np.random.seed(RSEED)

In [2]:
# Load the data paths
data_dir = "../data/encoded_data/"

# Load the raw data into a pandas dataframe
x_train = pd.read_csv(os.path.join(data_dir, "x_train_encoded.csv"))
y_train = pd.read_csv(os.path.join(data_dir, "y_train.csv"))
x_test = pd.read_csv(os.path.join(data_dir, "x_test_encoded.csv"))


# Load the shapes of the datasets
print(f"x_train shape: {x_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"x_test shape: {x_test.shape}")


x_train shape: (75973, 228)
y_train shape: (75973, 1)
x_test shape: (32567, 228)


In [3]:
# Check if x_train and x_test are standardscaled
print("First 5 rows of x_train:\n", x_train.head())
print("\nFirst 5 rows of x_test:\n", x_test.head())

First 5 rows of x_train:
       carID      year   mileage       tax       mpg  engineSize  \
0  1.437475 -0.501260  0.252019  0.353812 -2.795051    0.600708   
1  0.684586  0.872194 -0.834734  0.353812 -0.458753   -0.279931   
2 -1.441761  0.872194 -0.878739  0.353812 -0.907022   -0.279931   
3 -0.408772  0.414376 -0.628939  0.353812  0.681132   -1.160570   
4 -1.273236  0.872194 -0.998395  0.353812 -0.785349   -0.279931   

   paintQuality%  previousOwners  hasDamage  Brand_BMW  ...  model_ix35  \
0      -0.076836        1.382747        0.0  -0.328306  ...   -0.032263   
1      -0.701562       -0.686349        0.0  -0.328306  ...   -0.032263   
2      -0.413227        1.382747        0.0  -0.328306  ...   -0.032263   
3      -0.701562       -2.755444        0.0  -0.328306  ...   -0.032263   
4       1.557065        0.693048        0.0   3.045937  ...   -0.032263   

    model_x  transmission_manual  transmission_other  transmission_semi-auto  \
0 -0.005131            -1.157058        

# Filter Methods

In [4]:
# Check if there are univariate columns in x_train and x_test
x_train_var = x_train.var()
x_test_var = x_test.var()

print("Columns with zero variance in x_train:", x_train_var[x_train_var == 0].index.tolist())
print("Columns with zero variance in x_test:", x_test_var[x_test_var == 0].index.tolist())

Columns with zero variance in x_train: ['hasDamage']
Columns with zero variance in x_test: ['hasDamage', 'Brand_Škoda', 'model_A-Class', 'model_Ampera', 'model_B-Class', 'model_CL-Class', 'model_CLS-Class', 'model_Caddy', 'model_E-Class', 'model_Eos', 'model_GL-Class', 'model_GLE-Class', 'model_GLS-Class', 'model_Kona', 'model_Land Cruiser', 'model_M3', 'model_Puma', 'model_RAV4', 'model_RS', 'model_RS3', 'model_SLK-Class', 'model_Streetka', 'model_Tigra', 'model_Z', 'model_Zafira Toure', 'model_i20', 'model_i3', 'model_i40', 'model_i800', 'model_ix20', 'model_x', 'transmission_other']


In [5]:
# Drop the column hasDamage
x_train = x_train.drop(columns=['hasDamage'])
x_test = x_test.drop(columns=['hasDamage'])

In [6]:
# Check if there are univariate columns in x_train and x_test
x_train_var = x_train.var()
x_test_var = x_test.var()

print("Columns with zero variance in x_train:", x_train_var[x_train_var == 0].index.tolist())
print("Columns with zero variance in x_test:", x_test_var[x_test_var == 0].index.tolist())

Columns with zero variance in x_train: []
Columns with zero variance in x_test: ['Brand_Škoda', 'model_A-Class', 'model_Ampera', 'model_B-Class', 'model_CL-Class', 'model_CLS-Class', 'model_Caddy', 'model_E-Class', 'model_Eos', 'model_GL-Class', 'model_GLE-Class', 'model_GLS-Class', 'model_Kona', 'model_Land Cruiser', 'model_M3', 'model_Puma', 'model_RAV4', 'model_RS', 'model_RS3', 'model_SLK-Class', 'model_Streetka', 'model_Tigra', 'model_Z', 'model_Zafira Toure', 'model_i20', 'model_i3', 'model_i40', 'model_i800', 'model_ix20', 'model_x', 'transmission_other']


In [7]:
# Correlation Check
# Check the Spearman correlation of x_train features
cor_spearman = x_train.corr(method ='spearman')
cor_spearman


,carID,year,mileage,tax,mpg,engineSize,paintQuality%,previousOwners,Brand_BMW,Brand_Ford,...,model_ix35,model_x,transmission_manual,transmission_other,transmission_semi-auto,transmission_unknown,fuelType_electric,fuelType_hybrid,fuelType_other,fuelType_petrol
carID,1.000000,0.013909,-0.023906,-0.044472,-0.007543,-0.175209,-0.006625,-0.000204,-0.400493,-0.337123,...,-0.006873,-0.006241,0.119514,0.003041,-0.086603,0.002044,-0.003040,0.063921,0.029884,0.111417
year,0.013909,1.000000,-0.764944,0.299351,-0.294418,-0.038025,0.004943,0.001137,0.008097,-0.048212,...,-0.046974,0.000574,-0.157176,-0.000297,0.170923,0.006373,-0.006806,-0.012293,0.006229,0.102265
mileage,-0.023906,-0.764944,1.000000,-0.239283,0.302642,0.096860,-0.001428,0.003946,0.004588,0.033002,...,0.037008,-0.000962,0.134003,-0.000514,-0.143567,-0.005234,0.002695,0.016708,-0.004251,-0.191718
tax,-0.044472,0.299351,-0.239283,1.000000,-0.534477,0.145428,0.003854,-0.000603,0.048896,-0.019223,...,0.031768,-0.001973,-0.148339,0.001688,0.128650,0.000483,-0.012771,-0.195313,-0.012223,0.126810
mpg,-0.007543,-0.294418,0.302642,-0.534477,1.000000,-0.184750,0.001738,0.001661,-0.030062,0.122264,...,-0.020561,-0.000441,0.191721,-0.001749,-0.160045,-0.001550,0.011067,0.211073,0.020961,-0.319380
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
transmission_unknown,0.002044,0.006373,-0.005234,0.000483,-0.001550,-0.005914,0.000021,-0.002581,-0.001487,0.003182,...,0.013677,-0.000503,-0.113336,-0.000795,-0.051643,1.000000,-0.000711,-0.000921,-0.001701,0.002538
fuelType_electric,-0.003040,-0.006806,0.002695,-0.012771,0.011067,0.005095,0.002961,0.002576,-0.002382,0.008921,...,-0.000234,-0.000037,-0.008396,-0.000059,-0.003826,-0.000711,1.000000,-0.001260,-0.000341,-0.008218
fuelType_hybrid,0.063921,-0.012293,0.016708,-0.195313,0.211073,0.059450,-0.002325,-0.002012,-0.005660,-0.084584,...,-0.005604,-0.000891,-0.183934,-0.001409,-0.042001,-0.000921,-0.001260,1.000000,-0.008153,-0.196727
fuelType_other,0.029884,0.006229,-0.004251,-0.012223,0.020961,-0.012417,0.000426,-0.002656,0.008299,-0.025091,...,-0.001514,-0.000241,-0.039537,0.034265,-0.024746,-0.001701,-0.000341,-0.008153,1.000000,-0.053160


In [25]:
import numpy as np

# Sortiere alle Korrelationen (ohne Diagonale) nach Stärke
corr_pairs = (
    cor_spearman.where(~np.eye(cor_spearman.shape[0], dtype=bool))  # Diagonale ausschließen
    .abs()
    .stack()
    .sort_values(ascending=False)
)

print(corr_pairs.head(10))


year                    mileage                   0.764944
mileage                 year                      0.764944
model_C-Class           Brand_Mercedes-Benz       0.627711
Brand_Mercedes-Benz     model_C-Class             0.627711
transmission_manual     transmission_semi-auto    0.610038
transmission_semi-auto  transmission_manual       0.610038
engineSize              fuelType_petrol           0.609330
fuelType_petrol         engineSize                0.609330
carID                   Brand_Volkswagen          0.592879
Brand_Volkswagen        carID                     0.592879
dtype: float64


In [16]:
# Filter methods - Mutual Information
from sklearn.feature_selection import f_regression, mutual_info_regression, SelectKBest

X_filtered = SelectKBest(score_func=mutual_info_regression, k='all').fit(x_train, y_train)


In [38]:
scores = pd.DataFrame({
    'Feature': x_train.columns,
    'Score': X_filtered.scores_
}).sort_values('Score', ascending=False)

print("Top 10 features by score:\n", scores.head(10))


# Top 15 Features auswählen
top_features = scores.head(15)['Feature'].tolist()

selected_filter = scores[scores['Score'] > 0.01]['Feature']
x_train_reduced = x_train[selected_filter]

# count selected_filter
print("Number of selected features:", len(selected_filter))


Top 10 features by score:
                     Feature     Score
5                engineSize  0.396329
4                       mpg  0.383062
2                   mileage  0.343700
1                      year  0.339243
0                     carID  0.236491
219     transmission_manual  0.208458
3                       tax  0.185991
221  transmission_semi-auto  0.124746
11      Brand_Mercedes-Benz  0.110978
12               Brand_Opel  0.072719
Number of selected features: 36


## Wrapper Methods

In [41]:
# Mit RFE (Wrapper) validieren

from sklearn.feature_selection import RFE
from sklearn.linear_model import Ridge

model = Ridge(alpha=1.0)
rfe = RFE(model, n_features_to_select=10)
rfe.fit(x_train_reduced, y_train)

# als Liste ausgeben untereinander
selected_rfe = x_train_reduced.columns[rfe.support_].tolist()
print("RFE-selected features:")
for feature in selected_rfe:
    print("-", feature)

RFE-selected features:
- engineSize
- mpg
- mileage
- year
- transmission_manual
- Brand_Opel
- Brand_Ford
- Brand_Toyota
- Brand_Škoda
- Brand_Hyundai


In [43]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor
import numpy as np

rf = RandomForestRegressor(random_state=42)

base = -cross_val_score(rf, x_train, y_train, cv=5, scoring='neg_mean_absolute_error').mean()
reduced = -cross_val_score(rf, x_train_reduced[selected_rfe], y_train, cv=5, scoring='neg_mean_absolute_error').mean()

print(f"Baseline MAE: {base:.3f}")
print(f"After Feature Selection MAE: {reduced:.3f}")


Baseline MAE: 1421.924
After Feature Selection MAE: 1867.911
